In [2]:
import pandas as pd

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = pd.read_csv("player.csv")

In [ ]:
df.head()

In [ ]:
import numpy as np
from scipy import stats

In [ ]:
df["birthZ"] = stats.zscore(df["birthYear"] + (df["birthMonth"] - 1.0)/12.0 + (df["birthDay"] - 1.0)/30.0, nan_policy="omit")

In [ ]:
df["weightZ"] = stats.zscore(df["weight"], nan_policy="omit")
df["heightZ"] = stats.zscore(df["height"], nan_policy="omit")
df["batsN"] = df["bats"].apply(lambda b: 1.0 if b == 'R' else -1.0 if b == 'L' else 0.0)
df["throwsN"] = df["throws"].apply(lambda b: 1.0 if b == 'R' else -1.0 if b == 'L' else 0.0)


In [ ]:
df[["birthZ", "weightZ", "heightZ"]] = df[["birthZ", "weightZ", "heightZ"]].fillna(0.0)

In [ ]:
from sklearn.neighbors import NearestNeighbors

In [ ]:
features = ["birthZ", "heightZ", "weightZ", "batsN", "throwsN"]
nn_model = NearestNeighbors(n_neighbors=25)
nn_model.fit(df[features])



In [ ]:
def get_nearest_neighbors(id: str, n=25):
    seed = df[df["playerID"] == id][features]
    neighbor_indices = nn_model.kneighbors(seed, n, return_distance=False)
    return df.take(neighbor_indices[0])["id"]

In [ ]:
df.head()

In [ ]:
seed = df[df["playerID"] == "aaronha01"][features]
neighbor_indices = nn_model.kneighbors(seed, 25, return_distance=False)


In [ ]:
df.take(neighbor_indices[0])

In [ ]:
import joblib

joblib.dump(nn_model, "team_model.joblib")

In [ ]:
df.to_csv("features_db.csv", index=False)

In [ ]:
"aardsda01" in list(df["playerID"])